In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("taxi_type", "")
dbutils.widgets.text("year", "")
dbutils.widgets.text("month", "")

taxi_type = dbutils.widgets.get("taxi_type")
year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

In [0]:
from datetime import datetime
import pandas as pd
import pyspark.sql.functions as F

In [0]:
import sys

sys.path.append("../../src")

from dataDP.utils.logger import logger
from dataDP.config import DEFAULT_SRC_DEFINITIONS_TABLE
from dataDP.stg.create_stg_tables import populate_stg_tables_from_storage
from dataDP.table_management.definition import ColumnDefinition, TableDefinition, create_table_definition_from_dict

In [ ]:
stg_data_df = spark.table("workspace.stg.yellow_taxi")
# Uniformize column names
stg_data_df = stg_data_df.select([F.col(col).alias(col.replace("_utc", "")) for col in stg_data_df.columns])

In [0]:
yellow_taxi = TableDefinition(
    table_name="workspace.silver.yellow_taxi",
    columns=[
        ColumnDefinition("VendorID", "int", False ),
        ColumnDefinition("tpep_pickup_datetime", "timestamp", False),
        ColumnDefinition("tpep_dropoff_datetime", "timestamp", False),
        ColumnDefinition("passenger_count", "long", False, min_value = 1),
        ColumnDefinition("trip_distance", "double", False, custom_condition="trip_distance > 0"),
        ColumnDefinition("PULocationID", "integer", False, min_value = 1),
        ColumnDefinition("DOLocationID", "integer", False, min_value = 1),
        ColumnDefinition("fare_amount", "double", False, custom_condition="fare_amount > 0"),
        ColumnDefinition("total_amount", "double", False, min_value = 1),
    ],
    key_columns=["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime"],
)

In [ ]:
try:
    exist_df = spark.sql(f"SELECT * FROM {yellow_taxi.table_name} limit 1")
except:
    exist_df = spark.createDataFrame([], yellow_taxi.to_schema())

In [ ]:
silver_columns = [col.name for col in yellow_taxi.columns]
stg_df = stg_data_df.select(silver_columns)

# Check STG schema drift against silver definition
schema_report = yellow_taxi.compare_schema(stg_df)
print(schema_report)

In [ ]:
# Check Silver schema drift
schema_report = yellow_taxi.compare_schema(exist_df)
print(schema_report)

In [ ]:
ddl = yellow_taxi.generate_ddl()
print(ddl)

In [ ]:
constraints = yellow_taxi.generate_constraints()

In [ ]:
validate_df = yellow_taxi.validate_dataframe(stg_df)

In [ ]:
validate_with_reasons_df = yellow_taxi.validate_with_reasons(stg_df)

In [ ]:
yellow_taxi.ensure_version_sync()